# Validación predictiva con 2025 reservado (criterio C3)

Responde el criterio de éxito C3 y el criterio 4 de la rúbrica ("línea base implementada"): ¿predice el modelo bayesiano de medición mejor que el lineal jerárquico?

**Diseño.** Se entrena con 2015–2024 y se predice 2025. Es una validación temporal por bloque: el año de prueba no participa del ajuste, lo que evita la fuga de información de una partición aleatoria de observaciones del mismo establecimiento.

| Modelo | Descripción |
|---|---|
| M0 media global | Promedio nacional 2022–2024, igual para todos |
| M1 persistencia | Puntaje 2024 de cada establecimiento |
| M2 tendencia propia | Recta de cada establecimiento 2022–2024, extrapolada |
| M3 lineal jerárquico | Modelo mixto frecuentista, intercepto y pendiente aleatorios (línea base preespecificada) |
| M3b caída aleatoria | M3 con la caída de 2022 también aleatoria por escuela (línea base justa, agregada en la auditoría) |
| M4 bayesiano | Modelo de medición del notebook 06 |

Todos se evalúan sobre el mismo conjunto de establecimientos. El ajuste bayesiano tarda unos 6 minutos.

> **Revisión del 24-oct-2026 (auditoría de cuadernos y código).** Se agrega el modelo **M3b**: el mismo lineal jerárquico de M3, pero con la caída de 2022 también aleatoria por escuela. Es la línea base justa: M3 obliga a todas las escuelas a caer lo mismo en 2022, y M4 le ganaba sobre todo por eso.

**Salida esperada** (2.843 escuelas): M3 16,93; **M3b 15,66**; M4 15,65 (RMSE). Frente a M3, M4 mejora 1,29 [1,05; 1,52]. Frente a M3b, la diferencia es 0,02 [−0,08; 0,11]: **no hay ventaja predictiva del enfoque bayesiano** sobre un frecuentista bien especificado. El criterio C3 se cumple frente a la línea base preespecificada (M3), pero la lectura honesta es que la ventaja del modelo bayesiano está en la incertidumbre por escuela, calibrada (cobertura 0,945), y no en la precisión puntual. M3b tarda unos 20 segundos.

In [1]:
import sys, warnings
from pathlib import Path
for m in list(sys.modules):
    if m == "simce_panel" or m.startswith("simce_panel."):
        del sys.modules[m]
candidatos = [Path.cwd() / "simce_panel" / "src", Path.cwd().parent / "src", Path.cwd() / "src"]
sys.path.insert(0, str(next(p for p in candidatos if (p / "simce_panel" / "__init__.py").exists())))
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import simce_panel as sp
sp.configurar_logging()
cfg = sp.Config.desde_base(".")
P, B = sp.prediccion, sp.bayes

In [2]:
ancho = sp.io_datos.leer_csv(cfg.ruta_salida("simce_rbd_ancho_reducido_2015_2025.csv"))
ANIOS = [2015, 2016, 2017, 2018, 2022, 2023, 2024, 2025]
largo = pd.concat([pd.DataFrame({
    "rbd": ancho["rbd"], "cod_grupo_cat": ancho["cod_grupo_cat"], "anio": t,
    "y": pd.to_numeric(ancho[f"prom_mate2m_rbd_{t}"], errors="coerce"),
    "n": pd.to_numeric(ancho[f"nalu_mate2m_rbd_{t}"], errors="coerce")})
    for t in ANIOS]).dropna(subset=["y"])
ent, pru = P.preparar_particion(largo)
print("Establecimientos de prueba:", len(pru))

15:47:21 | INFO    | simce_panel.prediccion | Entrenamiento: 20520 obs.; prueba: 2964 establecimientos en 2025.


Establecimientos de prueba: 2964


## 1. Modelos de referencia (M0 a M3)

In [3]:
tabla = (P.predicciones_referencia(ent, pru)
         .join(P.prediccion_mixto(ent, pru))
         .join(P.prediccion_mixto(ent, pru, caida_aleatoria=True)))   # M3b
P.comparar(tabla)

,RMSE,MAE,sesgo,mejora_vs_M0_%,mejora_vs_M3_%
modelo,,,,,
M0 media global,41.08,31.82,-7.07,0.0,-140.2
M1 persistencia,17.92,13.69,-4.11,56.4,-4.8
M2 tendencia propia,23.48,17.89,-0.37,42.8,-37.3
M3 lineal jerárquico,17.10,13.10,-0.27,58.4,0.0
M3b caída aleatoria,15.84,12.08,-0.26,61.4,7.4


**Lectura.** M2 debería ser el peor de los modelos por establecimiento: extrapola una recta de tres puntos ruidosos. Es el problema que motiva el agrupamiento parcial: M3 lo resuelve parcialmente, y la pregunta es si M4 mejora sobre M3.

M3b muestra cuánto de la ventaja de M4 sobre M3 viene simplemente de permitir que cada escuela caiga distinto en 2022.

## 2. Modelo bayesiano de medición (M4)

Se ajusta sin 2025 (con al menos cuatro años por establecimiento, porque el año de prueba ya no está). La predicción incluye un intervalo que agrega el error de medición del año de prueba, más ancho para los colegios con pocos evaluados.

In [4]:
datos = B.preparar_datos(ent, col_n="n", min_anios=4)
mcmc = B.ajustar(datos, tendencia_pre=True, num_warmup=1000, num_samples=1000)
print(mcmc.diagnosticos)
n25 = pru.set_index("rbd")["n"]
tabla = tabla.join(P.prediccion_bayes(mcmc, datos, pru, n25))

15:47:38 | INFO    | simce_panel.bayes | 70 escuelas excluidas por no tener puntaje en 2022.


15:47:38 | INFO    | simce_panel.bayes | Datos del modelo: 2861 escuelas, 19763 observaciones.


15:47:39 | INFO    | jax._src.xla_bridge | Unable to initialize backend 'tpu': UNIMPLEMENTED: LoadPjrtPlugin is not implemented on windows yet.


15:50:04 | INFO    | simce_panel.bayes | Ajuste: {'segundos': 145.0, 'divergencias': 0, 'r_hat_max': 1.002, 'n_eff_min': 659}


{'segundos': 145.0, 'divergencias': 0, 'r_hat_max': 1.002, 'n_eff_min': 659}


## 3. Comparación

In [5]:
modelos = ["M0 media global", "M1 persistencia", "M2 tendencia propia",
           "M3 lineal jerárquico", "M3b caída aleatoria", "M4 bayesiano"]
comparacion = P.comparar(tabla, modelos=modelos)
print("Establecimientos evaluados:", comparacion.attrs["n"])
comparacion

Establecimientos evaluados: 2843


,RMSE,MAE,sesgo,mejora_vs_M0_%,mejora_vs_M3_%
modelo,,,,,
M0 media global,41.11,31.80,-6.97,0.0,-142.8
M1 persistencia,17.73,13.56,-4.16,56.9,-4.7
M2 tendencia propia,23.09,17.69,-0.33,43.8,-36.4
M3 lineal jerárquico,16.93,12.98,-0.30,58.8,0.0
M3b caída aleatoria,15.66,11.95,-0.26,61.9,7.5
M4 bayesiano,15.65,11.95,0.02,61.9,7.6


In [6]:
# ¿Es la mejora atribuible al azar de la muestra de prueba?
print("Frente a M3 :", P.diferencia_rmse_bootstrap(tabla, "M3 lineal jerárquico", "M4 bayesiano"))
print("Frente a M3b:", P.diferencia_rmse_bootstrap(tabla, "M3b caída aleatoria", "M4 bayesiano"))

Frente a M3 : {'diferencia_RMSE': 1.29, 'IC95': [1.05, 1.52], 'prob_candidato_mejor': 1.0, 'pct_establecimientos_mejor': 56.4, 'n': 2843}
Frente a M3b: {'diferencia_RMSE': 0.01, 'IC95': [-0.09, 0.11], 'prob_candidato_mejor': 0.569, 'pct_establecimientos_mejor': 50.4, 'n': 2843}


**Criterio C3.** Se cumple si el RMSE de M4 es menor que el de M3 y el intervalo bootstrap de la diferencia excluye el cero. Con la línea base preespecificada, se cumple.

**Lectura honesta (auditoría del 24-oct-2026).** Frente a M3b la diferencia es de unas centésimas y su intervalo incluye el cero: el modelo bayesiano predice igual que un frecuentista bien especificado. Su aporte no es predecir mejor, sino entregar para cada escuela una incertidumbre que está calibrada (sección 4). Así debe declararse en el informe.

## 4. Calibración del intervalo predictivo y desempeño por tamaño

In [7]:
t = tabla.dropna(subset=["M4_inf", "observado"]).join(n25.rename("n_2025"))
cobertura = ((t.observado >= t.M4_inf) & (t.observado <= t.M4_sup)).mean()
print(f"Cobertura del intervalo al 94 %: {cobertura:.3f} | ancho medio: "
      f"{(t.M4_sup - t.M4_inf).mean():.1f} puntos")
t["tamaño"] = pd.qcut(t["n_2025"], 3, labels=["pequeños", "medianos", "grandes"])
t.groupby("tamaño").apply(lambda g: pd.Series({
    m: np.sqrt(((g[m] - g.observado) ** 2).mean())
    for m in ["M2 tendencia propia", "M3 lineal jerárquico", "M3b caída aleatoria", "M4 bayesiano"]})).round(2)

Cobertura del intervalo al 94 %: 0.945 | ancho medio: 60.9 puntos


,M2 tendencia propia,M3 lineal jerárquico,M3b caída aleatoria,M4 bayesiano
tamaño,,,,
pequeños,26.96,19.63,18.46,18.19
medianos,22.40,16.83,15.24,15.35
grandes,19.09,13.73,12.63,12.84


**Lectura.** Una cobertura cercana a 0,94 indica que el intervalo está calibrado: dice la verdad sobre su propia incertidumbre. La comparación por tamaño muestra dónde rinde el agrupamiento parcial; se espera la mayor ganancia absoluta en los establecimientos pequeños, que son los que más se benefician de tomar fuerza del conjunto.

**Limitación.** Es un único año de prueba. La validación por bloques espaciales (comunas reservadas) complementaría esta prueba temporal en Capstone II.

In [8]:
tabla.to_csv(cfg.ruta_salida("validacion_predictiva_2025.csv"), sep=";", encoding="utf-8-sig")
comparacion.to_csv(cfg.ruta_salida("validacion_predictiva_resumen.csv"), sep=";", encoding="utf-8-sig")
print("Guardado.")

Guardado.
